# 第 0 课 · 环境与工具：让 PyTorch 跑在 Mac 的 GPU 上

这一课要搞懂三件事：

1. **uv 帮我们做了什么**：给这个文件夹单独装一个 Python 3.12 和一套库，不影响电脑上别的项目。
2. **张量（tensor）是什么**：带形状的多维数组。大模型里的一切，包括输入的文字、模型参数、中间结果，最后都是张量。
3. **设备（device）是什么**：张量放在哪块芯片上算。Mac 的 GPU 在 PyTorch 里叫 `mps`。

**怎么上**：从上往下一格一格运行（`Shift+Enter`）。带 `TODO` 的格子先填好再运行。全部跑完，把第 3 步的输出贴给我。

## 第 1 步 · 环境自检

运行下面这格，看四件事：Python 应该是 3.12，芯片架构应该是 `arm64`，最后两行 MPS 都应该是 `True`。

如果 Python 不是 3.12，说明右上角选错了内核，点它换成 `.venv`。

In [16]:
import platform
import time

import torch

print(f"Python  {platform.python_version()}")
print(f"芯片架构 {platform.machine()}")  # Apple 芯片是 arm64，Intel 芯片是 x86_64
print(f"PyTorch {torch.__version__}")
print(f"MPS 编进了这份 PyTorch: {torch.backends.mps.is_built()}")
print(f"MPS 现在能用: {torch.backends.mps.is_available()}")

Python  3.12.12
芯片架构 arm64
PyTorch 2.14.1
MPS 编进了这份 PyTorch: True
MPS 现在能用: True


### 选设备：接着上一格最后一行

上一格最后一行「MPS 现在能用」打印的，就是 `torch.backends.mps.is_available()` 的返回值。返回 `True`，这台机器能用 Mac 的 GPU，设备选 `"mps"`；返回 `False`，就只能退回 `"cpu"`。

为什么要写成判断，不直接写死 `device = "mps"`？因为同一份代码以后会拿到别的机器上跑，比如阶段 5 租的 NVIDIA 显卡服务器，那里没有 MPS。写成判断，换了机器也不会报错。

**什么时候没有 MPS？** MPS 是苹果 Metal 图形接口上的计算功能，PyTorch 靠它调用 Mac 的 GPU。下面这些情况 `is_available()` 都是 `False`：

| 情况 | 为什么 | 该用什么设备 |
|---|---|---|
| Windows / Linux 电脑 | Metal 只有苹果系统有 | 有 NVIDIA 显卡用 `"cuda"`，没有就 `"cpu"` |
| Intel 芯片的 Mac | 现在的 PyTorch（2.14.1）只出 Apple 芯片 Mac 的安装包，Intel Mac 装不上新版 | `"cpu"`，只能用旧版 PyTorch |
| M 系列 Mac，但装的是 x86 版 Python | 它靠 Rosetta 转译运行，PyTorch 以为自己在 Intel Mac 上。看上一格的芯片架构，显示 `x86_64` 就是这种情况 | 换成 arm64 版 Python。你这台的 anaconda 和 uv 装的 Python 都是 arm64，没这个问题 |
| macOS 太旧 | PyTorch 2.14.1 的 Mac 安装包要求 macOS 14 及以上 | 升级系统 |
| Mac 上的 Docker 容器、Linux 虚拟机 | 里面跑的是 Linux，碰不到 Metal | `"cpu"` |

上一格还打印了另一个函数 `is_built()`，两者的区别：

- `is_built()`：这份 PyTorch 编译的时候带没带 MPS 功能。`False` 说明装错了版本。
- `is_available()`：带了，而且这台机器此刻真能用。`is_built()` 是 `True`、它却是 `False`，多半是系统或硬件不支持。

In [17]:
# TODO 1：用 torch.backends.mps.is_available() 判断：能用就选 "mps"，不能用就选 "cpu"。
device = "mps" if torch.backends.mps.is_available() else "cpu"

assert device is not None, "先完成 TODO 1"
print(f"本节课用的设备: {device}")

本节课用的设备: mps


## 第 2 步 · 张量的三个基本属性

每个张量都带着三样东西：**shape**（形状）、**dtype**（每个数用什么格式存）、**device**（放在哪块芯片上）。

In [18]:
x = torch.randn(3, 4)
print("x =", x)
print("shape:", x.shape, " dtype:", x.dtype, " device:", x.device)

x = tensor([[-0.8318, -1.6659, -0.1650,  0.2599],
        [-0.0586,  1.3371, -1.1693, -0.8579],
        [-0.6612,  0.1014, -0.3113, -1.1874]])
shape: torch.Size([3, 4])  dtype: torch.float32  device: cpu


### 怎么把张量搬到别的设备

刚创建的张量默认在 CPU 上，上一格打印的 `device: cpu` 就是这个意思。要让 GPU 算它，得先把它搬过去。

每个张量都有一个 `.to()` 方法，括号里写目标设备，比如：

```python
w = torch.ones(2, 2)
w_gpu = w.to("mps")    # w_gpu 是放在 GPU 上的那一份
```

三个要点：

- `.to()` **不改动原来的张量**，而是返回一个新的。所以要用新名字接住结果，原来那个仍然在 CPU 上。直接写 `x_dev = x` 只是给同一个张量多起了个名字，它还在 CPU 上。
- 括号里可以写字符串 `"mps"`，但更好的是直接放 TODO 1 里的变量 `device`：换到没有 MPS 的机器上，`device` 是 `"cpu"`，这行照样能跑。
- 搬过去之后打印出来是 `mps:0`。`:0` 是 GPU 的编号，Mac 只有一块 GPU，所以总是 0。

In [19]:
# TODO 2：用 .to() 把 x 搬到 TODO 1 选好的 device 上，结果存成 x_dev。
x_dev = x.to(device)

assert x_dev is not None, "先完成 TODO 2"
assert x_dev.device.type == device, f"x_dev 还在 {x_dev.device.type} 上，没搬到 {device}。提示：看上面那格讲的 .to()"
print("x_dev 在:", x_dev.device)
print("x   还在:", x.device)

x_dev 在: mps:0
x   还在: cpu


## 第 3 步 · CPU 和 GPU 赛跑

用两个 4096×4096 的随机矩阵做乘法，分别在 CPU 和 GPU 上计时。

运行之前先猜一下：GPU 会是 CPU 的几倍速度？（双击这格，把猜测写在这里）

**我的猜测**：__100__ 倍

### 矩阵乘法怎么写

PyTorch 里矩阵乘法用 `@` 运算符：`a @ b` 就是矩阵 a 乘矩阵 b，和 `torch.matmul(a, b)` 是一回事。注意 `a * b` 不是矩阵乘法，它是两个矩阵对应位置的数各自相乘。

下面这格定义了两个工具函数，你只需要填 `bench` 里的 TODO 3：

- `sync(dev)`：等 GPU 把手上的活干完再往下走。
- `bench(a, b)`：先算一次热身，然后连着算 10 次，返回平均一次的耗时。

`bench` 不关心 a、b 在哪个设备上：传进来的是 CPU 上的矩阵，就在 CPU 上算；是 GPU 上的，就在 GPU 上算。

In [28]:
def sync(dev):
    # GPU 是异步的：Python 把命令发出去就往下走，不等它算完。
    # 所以计时前后都要等 GPU 把手上的活干完，否则测出来的时间是假的。
    if dev.type == "mps":
        torch.mps.synchronize()


def bench(a, b, repeat=10):
    """返回 a @ b 平均一次的耗时（秒）。"""
    a @ b  # 预热：第一次调用要做初始化，不计时
    sync(a.device)
    start = time.perf_counter()
    for _ in range(repeat):
        # TODO 3：用 @ 做一次矩阵乘法，a 乘 b，结果存成 c
        c = a @ b
    sync(a.device)
    assert c is not None, "先完成 TODO 3"
    return (time.perf_counter() - start) / repeat

In [30]:
n = 4096
a = torch.randn(n, n)
b = torch.randn(n, n)

t_cpu = bench(a, b)
t_dev = bench(a.to(device), b.to(device))

flops = 2 * n**3  # n×n 的矩阵乘一次，大约要做 2n³ 次浮点运算
print(f"{n}×{n} 矩阵乘法，平均一次：")
print(f"  CPU : {t_cpu * 1000:8.1f} ms   {flops / t_cpu / 1e12:.2f} TFLOPS")
print(f"  {device.upper():4}: {t_dev * 1000:8.1f} ms   {flops / t_dev / 1e12:.2f} TFLOPS")
print(f"  {device} 是 CPU 的 {t_cpu / t_dev:.1f} 倍速度")

4096×4096 矩阵乘法，平均一次：
  CPU :    107.3 ms   1.28 TFLOPS
  MPS :     43.1 ms   3.19 TFLOPS
  mps 是 CPU 的 2.5 倍速度


## 跑完想一想

- 实际倍数和你猜的差多少？
- 输出里的 TFLOPS 是什么意思？CPU 和 GPU 各达到了多少？
- `sync()` 里那句 `torch.mps.synchronize()` 是干什么的？

想不明白很正常，把输出贴给我，我们一起讲。

## 本课新词

| 词 | 意思 |
|---|---|
| tensor | 张量，带形状的多维数组 |
| shape | 每一维有多长，比如 `(3, 4)` 是 3 行 4 列 |
| dtype | 每个数用什么格式存，默认 `float32`，每个数占 4 字节 |
| device | 张量所在的芯片：`cpu`、`mps`（Mac GPU）、`cuda`（NVIDIA GPU） |
| FLOPS | 每秒做多少次浮点运算；1 TFLOPS = 每秒一万亿次 |

作业在同目录的 `homework.ipynb`。